In [1]:
import mlflow
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from torch.utils.data import DataLoader, Dataset
from torchinfo import summary
from tqdm.notebook import tqdm
from transformers import AutoModel, AutoTokenizer
from transformers.models.bert.modeling_bert import BertModel
from mlflow.models import infer_signature

# Types
from transformers.models.bert.tokenization_bert_fast import BertTokenizerFast

import matplotlib.pyplot as plt
from src.config import MPL_STYLE_DIR
from src.db import PBWarehouse
from src.models import Tweet

mlflow.set_tracking_uri("http://192.168.100.203:5000")
mlflow.set_experiment("[CAPSTONE-2] bert-finetuning")
plt.style.use(MPL_STYLE_DIR / "iragca_ml.mplstyle")

warehouse = PBWarehouse()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

2025-07-06 14:19:43.194 | INFO     | src.config:<module>:25 - Loaded environment variables from /home/iragca/Documents/github/capstone-project-2/.env
2025-07-06 14:19:43.194 | INFO     | src.config:<module>:59 - PROJECT_ROOT: /home/iragca/Documents/github/capstone-project-2
2025-07-06 14:19:43.195 | INFO     | src.config:<module>:60 - DATA_DIR: /home/iragca/Documents/github/capstone-project-2/data


device(type='cuda')

In [2]:
data = warehouse.client.collection("tweets_v2").get_full_list(
    query_params={
        "filter": "(has_blm_hashtag = true || is_reply_to_blm = true) && is_annotated = true"
    }
)
tweets: list[Tweet] = [Tweet(**r.__dict__) for r in data]

In [3]:
df = pl.DataFrame(
    [t.model_dump() for t in tweets],
    schema={
        "tweet_id": pl.Utf8,
        "text": pl.Utf8,
        "status_link": pl.Utf8,
        "user_id": pl.Utf8,
        "is_extremist": pl.Boolean,
        "is_annotated": pl.Boolean,
        "bookmark_count": pl.Int64,
        "views": pl.Int64,
        "retweet_count": pl.Int64,
        "favorite_count": pl.Int64,
        "reply_count": pl.Int64,
        "quote_count": pl.Int64,
        "in_reply_to_status_id": pl.Utf8,
        "conversation_id": pl.Utf8,
        "retweet_tweet_id": pl.Utf8,
        "quoted_status_id": pl.Utf8,
        "community_note": pl.Utf8,
        "language": pl.Utf8,
        "source": pl.Utf8,
        "creation_date": pl.Utf8,
        "has_blm_hashtag": pl.Boolean,
        "fetched_replies": pl.Boolean,
        "is_reply_to_blm": pl.Boolean,
    },
).with_columns(
    pl.col("creation_date").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S%.3fZ")
)
df

tweet_id,text,status_link,user_id,is_extremist,is_annotated,bookmark_count,views,retweet_count,favorite_count,reply_count,quote_count,in_reply_to_status_id,conversation_id,retweet_tweet_id,quoted_status_id,community_note,language,source,creation_date,has_blm_hashtag,fetched_replies,is_reply_to_blm
str,str,str,str,bool,bool,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,str,datetime[ms],bool,bool,bool
"""1286440846779224064""","""finally. thank you #BLM""","""https://x.com/JaneGPhoto/statu…","""19804476""",false,true,0,0,0,0,0,0,"""""","""1286440846779224064""","""""","""1286440670367014914""","""""","""en""","""Twitter Web App""",2020-07-23 23:19:33,true,true,false
"""1286436325017694215""","""Dr. Fauci throws out the first…","""https://x.com/KevinBarryKC/sta…","""942852873762824192""",false,true,0,0,2,5,0,0,"""""","""1286436325017694215""","""""","""""","""""","""en""","""Twitter for iPhone""",2020-07-23 23:01:35,true,true,false
"""1286446247935709187""","""Ok so @NiagaraRegion just fini…","""https://x.com/GrantRants/statu…","""171610230""",false,true,0,0,2,16,1,0,"""""","""1286446247935709187""","""""","""""","""""","""en""","""Twitter Web App""",2020-07-23 23:41:01,true,true,false
"""1286446875063664640""","""@RyanShead Hey!? A Bosox cap? …","""https://x.com/ReiRei720dxcute/…","""1232338981397729281""",false,true,0,0,0,3,1,0,"""1286436366063026176""","""1286436366063026176""","""""","""""","""""","""en""","""""",2020-07-23 23:43:30,false,true,true
"""1286446607462957056""","""Imma just set this right here …","""https://x.com/SheTalksSeXXX/st…","""1197060347062104064""",false,true,0,0,0,2,0,0,"""""","""1286446607462957056""","""""","""""","""""","""en""","""Twitter for Android""",2020-07-23 23:42:26,true,true,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""1286443740639395840""","""So proud of my daughter who’s …","""https://x.com/JinnieSpiegler/s…","""154714175""",false,true,0,0,0,3,0,0,"""""","""1286443740639395840""","""""","""""","""""","""en""","""Twitter for iPhone""",2020-07-23 23:31:03,true,true,false
"""1286448066086633472""","""I had held off sharing this ph…","""https://x.com/GarthGodsman/sta…","""16073596""",true,true,0,0,1,0,0,0,"""""","""1286448066086633472""","""""","""""","""""","""en""","""Twitter Web App""",2020-07-23 23:48:14,true,true,false
"""1286442153758138368""","""#NowPlaying Gotta Be - Reflexi…","""https://x.com/TunitupRadio/sta…","""317869483""",false,true,0,0,1,0,0,0,"""""","""1286442153758138368""","""""","""""","""""","""en""","""TUN-IT-UP Radio""",2020-07-23 23:24:45,true,true,false


In [4]:
train_df = df.select(["text", "is_extremist"])
dataset = mlflow.data.from_pandas(
    train_df.to_pandas(), name="BLM Tweets", targets="is_extremist"
)

train_df

text,is_extremist
str,bool
"""finally. thank you #BLM""",false
"""Dr. Fauci throws out the first…",false
"""Ok so @NiagaraRegion just fini…",false
"""@RyanShead Hey!? A Bosox cap? …",false
"""Imma just set this right here …",false
…,…
"""So proud of my daughter who’s …",false
"""I had held off sharing this ph…",true
"""#NowPlaying Gotta Be - Reflexi…",false


In [5]:
X_validation, X_holdout, y_validation, y_holdout = train_test_split(
    train_df["text"],
    train_df["is_extremist"],
    test_size=0.2,
    random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(
    X_validation,
    y_validation,
    test_size=0.2,
    random_state=42,
)

print(
    f"Train: {len(X_train)}, Test: {len(X_test)}, Validation: {len(X_validation)}, Holdout: {len(X_holdout)}"
)
print(
    f"Train ratio: {len(X_train) / len(df):.2f}, Test ratio: {len(X_test) / len(df):.2f}, Validation ratio: {len(X_validation) / len(df):.2f}, Holdout ratio: {len(X_holdout) / len(df):.2f}"
)

Train: 44, Test: 12, Validation: 56, Holdout: 14
Train ratio: 0.63, Test ratio: 0.17, Validation ratio: 0.80, Holdout ratio: 0.20


In [6]:
bert_model_name = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModel.from_pretrained(bert_model_name)

In [7]:
class ExtremistDataset(Dataset):
    def __init__(self, X: pl.Series, y: pl.Series, bert_tokenizer: BertTokenizerFast):
        assert isinstance(X, pl.Series), "X must be a polars DataFrame"
        assert isinstance(y, pl.Series), "y must be a polars DataFrame"
        assert len(X) == len(y), "X and y must have the same length"
        assert "text" == X.name, "X must contain a 'text' column for the tweets"
        assert "is_extremist" == y.name, "y must contain an 'is_extremist' column"
        assert isinstance(bert_tokenizer, BertTokenizerFast), (
            "tokenizer must be an instance of AutoTokenizer"
        )
        self.X = [
            bert_tokenizer(
                x,
                max_length=100,
                truncation=True,
                padding="max_length",
                return_tensors="pt",
            )
            for x in X
        ]

        self.y = y.to_torch().to(torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        return x, y


training_dataset = ExtremistDataset(X_train, y_train, tokenizer)
validation_dataset = ExtremistDataset(X_test, y_test, tokenizer)
holdout_dataset = ExtremistDataset(X_holdout, y_holdout, tokenizer)

In [8]:
params = {
    "BATCH_SIZE": 32,
    "EPOCHS": 10,
    "LEARNING_RATE": 1e-4,
    "DROPOUT_RATE": 0.25,
}

In [9]:
train_dataloader = DataLoader(
    training_dataset, batch_size=params["BATCH_SIZE"], shuffle=True
)
validation_dataloader = DataLoader(
    validation_dataset, batch_size=params["BATCH_SIZE"], shuffle=False
)
holdout_dataloader = DataLoader(
    holdout_dataset, batch_size=params["BATCH_SIZE"], shuffle=False
)

# Training

In [10]:
class ExtremismDetector(nn.Module):
    def __init__(self, bert_model: BertModel, dropout_rate: float):
        super(ExtremismDetector, self).__init__()
        assert isinstance(bert_model, BertModel), (
            "bert_model must be an instance of AutoModel"
        )

        self.bert = bert_model
        self.linear1 = nn.Linear(bert_model.config.hidden_size, 384)
        self.dropout = nn.Dropout(dropout_rate)
        self.linear2 = nn.Linear(384, 1)
        self.activation = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        pooled_output = self.bert(
            input_ids=input_ids, attention_mask=attention_mask, return_dict=False
        )[0][:, 0]

        h = self.linear1(pooled_output)
        h = self.dropout(h)
        h = self.linear2(h)
        h = self.activation(h)

        return h

In [11]:
for param in bert_model.parameters():
    param.requires_grad = False

model = ExtremismDetector(bert_model, params["DROPOUT_RATE"])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=params["LEARNING_RATE"])


input_data = {
    "input_ids": training_dataset[0][0]["input_ids"],
    "attention_mask": training_dataset[0][0]["attention_mask"],
}

example_output = torch.tensor([0.95])  # e.g., probability output
# Convert to numpy for signature inference
example_input_numpy = {
    k: v.numpy() for k, v in training_dataset[0][0].items()
}
example_output_numpy = example_output.numpy()

# Create the signature
signature = infer_signature(example_input_numpy, example_output_numpy)

In [12]:
with mlflow.start_run(log_system_metrics=True) as run:
    mlflow.log_params(params)
    mlflow.log_input(dataset, context="training", tags={"source": "Twitter / X"})
    mlflow.set_tag("purpose", "configuring")
    mlflow.set_tag("framework", "pytorch")
    mlflow.set_tag("task-type", "text classification")
    mlflow.set_tag("task", "extremism detection")

    with open("model_summary.txt", "w") as f:
        f.write(
            str(
                summary(
                    model,
                    device=str(device),
                    input_data=input_data,
                    col_names=[
                        "input_size",
                        "output_size",
                        "num_params",
                        "params_percent",
                        "mult_adds",
                        "kernel_size",
                    ],
                )
            )
        )

    model.to("cpu")
    with open("model_architecture.txt", "w") as f:
        f.write(str(model))

    mlflow.pytorch.log_model(
        model,
        name="Sarcasm Detector",
        signature=signature,
        extra_files=["model_summary.txt", "model_architecture.txt"],
    )
    model.to(device)
    bert_model.to(device)
    for epoch in tqdm(range(params["EPOCHS"]), desc="Training epochs", unit="epoch"):
        ## Train and validate the model
        model.train()
        total_loss_train = 0
        total_loss_validation = 0

        all_preds_train = []
        all_labels_train = []

        all_preds_validation = []
        all_labels_validation = []

        for inputs, labels in tqdm(
            train_dataloader,
            desc=f"Epoch {epoch + 1}/{params['EPOCHS']} - Training batches",
            unit="batch",
            leave=False,
        ):
            inputs = inputs.to(device)
            labels = labels.to(device)

            predictions = model(
                input_ids=inputs["input_ids"].squeeze(1),
                attention_mask=inputs["attention_mask"].squeeze(1),
            ).squeeze(1)

            batch_loss = criterion(predictions, labels)
            total_loss_train += batch_loss.item()

            all_preds_train.extend(predictions.round().detach().cpu().numpy())
            all_labels_train.extend(labels.detach().cpu().numpy())

            batch_loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        with torch.no_grad():
            model.eval()
            for inputs, labels in tqdm(
                validation_dataloader,
                desc=f"Epoch {epoch + 1}/{params['EPOCHS']} - Validation batches",
                unit="batch",
                leave=False,
            ):
                inputs = inputs.to(device)
                labels = labels.to(device)

                predictions = model(
                    input_ids=inputs["input_ids"].squeeze(1),
                    attention_mask=inputs["attention_mask"].squeeze(1),
                ).squeeze(1)

                batch_loss = criterion(predictions, labels)
                total_loss_validation += batch_loss.item()

                all_preds_validation.extend(predictions.round().detach().cpu().numpy())
                all_labels_validation.extend(labels.detach().cpu().numpy())

        mlflow.log_metrics(
            {
                "train_loss": total_loss_train / len(train_dataloader),
                "train_accuracy": accuracy_score(all_labels_train, all_preds_train),
                "train_f1": f1_score(
                    all_labels_train, all_preds_train, zero_division=0
                ),
                "train_precision": precision_score(
                    all_labels_train, all_preds_train, zero_division=0
                ),
                "train_recall": recall_score(
                    all_labels_train, all_preds_train, zero_division=0
                ),
                "validation_loss": total_loss_validation / len(validation_dataloader),
                "validation_accuracy": accuracy_score(
                    all_labels_validation, all_preds_validation
                ),
                "validation_precision": precision_score(
                    all_labels_validation, all_preds_validation, zero_division=0
                ),
                "validation_recall": recall_score(
                    all_labels_validation, all_preds_validation, zero_division=0
                ),
                "validation_f1": f1_score(
                    all_labels_validation, all_preds_validation, zero_division=0
                ),
            },
            step=epoch,
        )


2025/07/06 14:19:46 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2025/07/06 14:19:46 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.


2025/07/06 14:19:52 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/07/06 14:19:56 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.


Training epochs:   0%|          | 0/10 [00:00<?, ?epoch/s]

Epoch 1/10 - Training batches:   0%|          | 0/2 [00:00<?, ?batch/s]

Epoch 1/10 - Validation batches:   0%|          | 0/1 [00:00<?, ?batch/s]

...

Epoch 9/10 - Validation batches:   0%|          | 0/1 [00:00<?, ?batch/s]

Epoch 10/10 - Training batches:   0%|          | 0/2 [00:00<?, ?batch/s]

Epoch 10/10 - Validation batches:   0%|          | 0/1 [00:00<?, ?batch/s]

2025/07/06 14:20:08 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2025/07/06 14:20:08 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run classy-fly-276 at: http://192.168.100.203:5000/#/experiments/5/runs/32cadf1ba55e4a37b7d442b3a4fb4d25
🧪 View experiment at: http://192.168.100.203:5000/#/experiments/5


# Testing

In [13]:
with mlflow.start_run(log_system_metrics=True) as run:
    mlflow.log_params(params)
    mlflow.log_input(dataset, context="testing", tags={"source": "Twitter / X"})
    mlflow.set_tag("purpose", "configuring")
    mlflow.set_tag("framework", "pytorch")
    mlflow.set_tag("task-type", "text classification")
    mlflow.set_tag("task", "extremism detection")

    model.to(device)
    with open("model_summary.txt", "w") as f:
        f.write(
            str(
                summary(
                    model,
                    device=str(device),
                    input_data=input_data,
                    col_names=[
                        "input_size",
                        "output_size",
                        "num_params",
                        "params_percent",
                        "mult_adds",
                        "kernel_size",
                    ],
                )
            )
        )

    model.to("cpu")
    with open("model_architecture.txt", "w") as f:
        f.write(str(model))


    mlflow.pytorch.log_model(
        model,
        name="Sarcasm Detector",
        signature=signature,
        extra_files=["model_summary.txt", "model_architecture.txt"],
    )

    model.to(device)
    bert_model.to(device)

    ## Evaluate on holdout set
    test_loss = 0.0
    all_preds_holdout = []
    all_labels_holdout = []

    with torch.no_grad():
        model.eval()
        for inputs, labels in tqdm(
            holdout_dataloader,
            desc="Holdout batches",
            unit="batch",
            leave=False,
        ):
            inputs = inputs.to(device)
            labels = labels.to(device)

            predictions = model(
                input_ids=inputs["input_ids"].squeeze(1),
                attention_mask=inputs["attention_mask"].squeeze(1),
            ).squeeze(1)

            batch_loss = criterion(predictions, labels)
            test_loss += batch_loss.item()

            all_preds_holdout.extend(predictions.round().detach().cpu().numpy())
            all_labels_holdout.extend(labels.detach().cpu().numpy())

        mlflow.log_metrics(
            {
                "holdout_loss": test_loss / len(holdout_dataloader),
                "holdout_accuracy": accuracy_score(
                    all_labels_holdout, all_preds_holdout
                ),
                "holdout_precision": precision_score(
                    all_labels_holdout, all_preds_holdout, zero_division=0
                ),
                "holdout_recall": recall_score(
                    all_labels_holdout, all_preds_holdout, zero_division=0
                ),
                "holdout_f1": f1_score(
                    all_labels_holdout, all_preds_holdout, zero_division=0
                ),
            }
        )


2025/07/06 14:20:08 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2025/07/06 14:20:08 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.


2025/07/06 14:20:13 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/07/06 14:20:18 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.


Holdout batches:   0%|          | 0/1 [00:00<?, ?batch/s]

2025/07/06 14:20:21 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2025/07/06 14:20:21 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run agreeable-sponge-893 at: http://192.168.100.203:5000/#/experiments/5/runs/51aa5bf34313499ca3b11ea22a8aed3f
🧪 View experiment at: http://192.168.100.203:5000/#/experiments/5
